# Football Analysis — Tracking GPU tự động (Colab)

Chỉ chạy **YOLO + BoT-SORT + ReID** trên GPU. Khi bạn upload video trên **web local**, backend tự gửi video sang Colab → tracking → tải stub về → pipeline còn lại chạy trên máy.

## Chuẩn bị (một lần)
1. **Runtime → Change runtime type → T4 GPU**
2. [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) (miễn phí)
3. `best.pt` trên Drive: `MyDrive/football_analysis/best.pt`

## Sau khi chạy cell server
Copy URL ngrok in ra. Trên máy local, tạo file `.env` ở thư mục gốc repo (hoặc set biến môi trường trước khi chạy backend):

```
COLAB_TRACKING_URL=https://xxxx.ngrok-free.app
```

Rồi chạy backend + frontend **local** (không cần `VITE_API_BASE` trỏ Colab):
```bash
uvicorn api.main_api:app --reload --port 8000
cd frontend && npm run dev
```

**Giữ tab Colab mở** trong lúc phân tích. Upload video trên web → Colab tự nhận và tracking.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/PhuongAnh777/football_analysis.git"
BRANCH = "Collab"  # đổi "main" nếu cần

%cd /content

if os.path.isdir("football_analysis/.git"):
    %cd football_analysis
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !rm -rf football_analysis
    !git clone -b {BRANCH} {REPO_URL}
    %cd football_analysis

!git branch --show-current
!ls api/tracking_server.py scripts/run_tracking.py

In [ ]:
# Colab đã có torch+CUDA — không cài lại torch
!pip install -q fastapi "uvicorn[standard]" python-multipart aiofiles requests \
    ultralytics opencv-python numpy pandas scikit-learn scipy matplotlib pyngrok

In [ ]:
import os
from google.colab import drive

DRIVE_MODEL = "/content/drive/MyDrive/football_analysis/best.pt"
MODEL_PATH = "models/best.pt"

os.makedirs("models", exist_ok=True)
os.makedirs("input_videos", exist_ok=True)
os.makedirs("stubs", exist_ok=True)

drive.mount("/content/drive")

if os.path.exists(DRIVE_MODEL):
    !cp "{DRIVE_MODEL}" "{MODEL_PATH}"
    print("Model OK:", MODEL_PATH)
else:
    from google.colab import files
    print("Không thấy model trên Drive. Upload best.pt:")
    uploaded = files.upload()
    !mv best.pt models/best.pt
    print("Model OK:", MODEL_PATH)

In [ ]:
import os
import socket
import time
import threading
import traceback
import subprocess
import urllib.request
import json

from pyngrok import ngrok
import uvicorn

# Paste authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ""  # ← điền token

if not NGROK_AUTHTOKEN:
    raise ValueError("Hãy điền NGROK_AUTHTOKEN trước khi chạy cell này.")

PORT = 8001
_server_error = []

# ── 1. Kiểm tra import trước khi khởi động ───────────────────────────────
print("Kiểm tra import api.tracking_server...", end="", flush=True)
try:
    import importlib
    import api.tracking_server as _m
    importlib.reload(_m)
    print(" OK")
except Exception:
    raise RuntimeError(
        f"Import api.tracking_server thất bại:\n{traceback.format_exc()}\n\n"
        "Kiểm tra: pip install các package còn thiếu rồi chạy lại."
    )

# ── 2. Kill port cũ + đóng tunnel ngrok cũ (tránh 'address already in use') ─
try:
    if subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True, timeout=5).returncode == 0:
        print(f"Đã kill process cũ trên port {PORT}")
        time.sleep(1)
except Exception:
    pass

try:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
        print(f"Đã đóng tunnel cũ: {t.public_url}")
except Exception:
    pass

def _run_server():
    try:
        uvicorn.run(
            "api.tracking_server:app",
            host="0.0.0.0",
            port=PORT,
            log_level="info",
        )
    except Exception as e:
        _server_error.append(traceback.format_exc())
        print(f"\n[server-thread] CRASH: {e}", flush=True)

# ── 3. Khởi động uvicorn ─────────────────────────────────────────────────
threading.Thread(target=_run_server, daemon=True).start()

print(f"Đang chờ uvicorn bind port {PORT}...", end="", flush=True)
for _i in range(25):
    time.sleep(1)
    if _server_error:
        raise RuntimeError(
            f"Server crash khi khởi động:\n{_server_error[0]}\n\n"
            "Thử: (1) Runtime → Restart runtime → chạy lại từ Cell 2\n"
            "     (2) Kiểm tra có lỗi import ở cell trên không"
        )
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=0.5):
            print(f" port OK ({_i+1}s)", flush=True)
            break
    except OSError:
        print(".", end="", flush=True)
else:
    raise RuntimeError(
        f"uvicorn không bind port {PORT} sau 25s.\n"
        "Thử: Runtime → Restart runtime → chạy lại từ Cell 2"
    )

# ── 4. Kiểm tra /api/health cục bộ trước khi mở ngrok ─────────────────────
print("Kiểm tra /api/health cục bộ...", end="", flush=True)
_last_health_err = None
for _j in range(10):
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/health")
        with urllib.request.urlopen(req, timeout=3) as r:
            _hdata = json.loads(r.read())
            print(f" OK → {_hdata}")
            break
    except Exception as _e:
        _last_health_err = _e
        print(".", end="", flush=True)
        time.sleep(1)
else:
    raise RuntimeError(
        "uvicorn đang chạy nhưng /api/health không phản hồi sau 10s.\n"
        f"Lỗi cuối: {_last_health_err}\nXem log uvicorn bên trên để biết chi tiết."
    )

# ── 5. Tạo ngrok tunnel SAU khi server đã sẵn sàng ──────────────────────
ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_tunnel = ngrok.connect(PORT, bind_tls=True)
public_url = public_tunnel.public_url.rstrip("/")

print("=" * 60)
print("Colab Tracking API:", public_url)
print("Health check:      ", f"{public_url}/api/health")
print()
print("Trên máy local, thêm vào file .env:")
print(f"  COLAB_TRACKING_URL={public_url}")
print()
print("Sau đó restart backend:  uvicorn api.main_api:app --reload --port 8000")
print("Giữ tab Colab mở trong lúc phân tích.")
print("=" * 60)

In [ ]:
# Kiểm tra server qua ngrok (chạy sau cell server)
import time
import requests

base = public_url
headers = {"ngrok-skip-browser-warning": "true"}
data = None
_last_err = None

print("Đang chờ ngrok + server...", end="", flush=True)
for attempt in range(15):
    time.sleep(2)
    try:
        h = requests.get(f"{base}/api/health", headers=headers, timeout=10)
        if h.status_code != 200:
            print(".", end="", flush=True)
            continue
        if not h.text.strip():
            _last_err = "API trả về body rỗng — server Colab có thể chưa khởi động"
            print(".", end="", flush=True)
            continue
        try:
            data = h.json()
        except requests.exceptions.JSONDecodeError:
            preview = h.text[:120].replace("\n", " ")
            _last_err = f"API không trả JSON (có thể ngrok warning page): {preview!r}"
            print(".", end="", flush=True)
            continue
        print(f"\nhealth: {h.status_code} {data}")
        break
    except requests.RequestException as e:
        _last_err = str(e)
        print(".", end="", flush=True)
else:
    raise RuntimeError(
        "Server không phản hồi JSON hợp lệ sau 30s.\n"
        f"Lỗi cuối: {_last_err}\n"
        "Kiểm tra: (1) Cell server đã chạy xong chưa? "
        "(2) NGROK_AUTHTOKEN hợp lệ? "
        "(3) Có lỗi 'address already in use' ở cell server không?"
    )

assert data.get("model_loaded"), "Thiếu models/best.pt — chạy lại cell copy model"
assert data.get("gpu_available"), "Chưa bật GPU — Runtime → Change runtime type → T4 GPU"
print("✓ Sẵn sàng nhận video từ web local")

## (Tùy chọn) Chạy tracking thủ công

Nếu không dùng web, vẫn có thể đặt video trên Drive và chạy cell dưới.

In [ ]:
# Bỏ qua nếu chỉ dùng web → Colab tự nhận video qua API
import os
from google.colab import drive

DRIVE_VIDEO = "/content/drive/MyDrive/football_analysis/input_video.mp4"
VIDEO_PATH = "input_videos/input_video.mp4"

if os.path.exists(DRIVE_VIDEO):
    !cp "{DRIVE_VIDEO}" "{VIDEO_PATH}"
    !python scripts/run_tracking.py "{VIDEO_PATH}"
    print("Stub:", "stubs/track_stubs.pkl")
else:
    print("Không có video trên Drive — dùng upload web hoặc upload file ở đây.")